# 第二课：NumPy 数据类型 `dtype`

本课依据菜鸟教程的 [NumPy 数据类型](https://www.runoob.com/numpy/numpy-dtype.html) 整理，并补充类型推断、类型转换、内存占用和常见陷阱。

学完后，你应该能够：

- 看懂数组的 `dtype`
- 创建数组时指定数据类型
- 使用 `astype()` 转换类型
- 理解 NumPy 为什么会自动统一数组类型
- 避免整数溢出、浮点精度和字符串截断等常见问题

## 1. 什么是 `dtype`？

普通 NumPy 数组中的元素通常使用同一种数据类型。`dtype`（data type）描述了每个元素在内存中如何存储。

它不仅说明元素是整数、浮点数还是字符串，还包含：

- 每个元素占多少字节
- 数值是否有符号
- 浮点数的精度
- 必要时的字节顺序和结构化字段

常见类型包括：

| 类型 | 含义 | 常见范围或特点 |
|---|---|---|
| `bool_` | 布尔值 | `True`、`False` |
| `int8` | 8 位有符号整数 | -128 到 127 |
| `uint8` | 8 位无符号整数 | 0 到 255 |
| `int16` / `int32` / `int64` | 不同位数的整数 | 位数越大，范围越大 |
| `float16` / `float32` / `float64` | 浮点数 | 位数越大，通常精度越高 |
| `complex64` / `complex128` | 复数 | 保存实部和虚部 |
| `str_` | Unicode 字符串 | 长度通常会编码在 dtype 中 |
| `datetime64` | 日期时间 | 支持按时间单位存储和运算 |

> 不要死记所有类型。现阶段重点掌握 `bool`、整数、浮点数、复数和字符串。

In [1]:
import numpy as np

print("NumPy 版本：", np.__version__)

NumPy 版本： 2.5.3


## 2. 查看数组的数据类型

创建数组时，如果没有指定 `dtype`，NumPy 会根据输入数据自动推断。

In [2]:
integers = np.array([1, 2, 3])
decimals = np.array([1.0, 2.5, 3.8])
booleans = np.array([True, False, True])
complex_numbers = np.array([1 + 2j, 3 + 4j])
words = np.array(["NumPy", "Python"])

for name, array in [
    ("整数", integers),
    ("浮点数", decimals),
    ("布尔值", booleans),
    ("复数", complex_numbers),
    ("字符串", words),
]:
    print(f"{name:4}：{str(array):22} dtype = {array.dtype}")

整数  ：[1 2 3]                dtype = int64
浮点数 ：[1.  2.5 3.8]          dtype = float64
布尔值 ：[ True False  True]    dtype = bool
复数  ：[1.+2.j 3.+4.j]        dtype = complex128
字符串 ：['NumPy' 'Python']     dtype = <U6


你的整数结果可能显示为 `int32` 或 `int64`，这可能与平台和创建方式有关，属于正常现象。需要固定范围或内存大小时，应显式指定类型。

## 3. 创建数组时指定类型

可以把 NumPy 类型对象传给 `dtype`。这是最清晰、最推荐的写法。

In [ ]:
a = np.array([1, 2, 3], dtype=np.int32)
b = np.array([1, 2, 3], dtype=np.float64)
c = np.array([1, 2, 3], dtype=np.complex128)

print(a, a.dtype)
print(b, b.dtype)
print(c, c.dtype)

### dtype 字符代码

菜鸟教程还介绍了简写字符：`i` 表示有符号整数，`u` 表示无符号整数，`f` 表示浮点数，`c` 表示复数。后面的数字表示字节数。

例如，`i4` 是 4 字节整数，也就是 32 位整数；`f8` 是 8 字节浮点数，也就是 64 位浮点数。学习时建议优先使用 `np.int32`、`np.float64` 这类可读性更好的名称。

In [ ]:
short_int = np.array([10, 20, 30], dtype="i2")
single_float = np.array([1, 2, 3], dtype="f4")

print(short_int, short_int.dtype)
print(single_float, single_float.dtype)
print("i2 与 np.int16 是否相同：", np.dtype("i2") == np.dtype(np.int16))

## 4. 混合数据与类型提升

一个普通 `ndarray` 通常只能有一种 `dtype`。输入中出现不同类型时，NumPy 会寻找一种能够统一保存所有元素的类型，这常被称为**类型提升**。

In [ ]:
mixed_number = np.array([1, 2.5, 3])
mixed_text = np.array([1, 2.5, "hello"])

print(mixed_number, mixed_number.dtype)
print(mixed_text, mixed_text.dtype)

第一组中，整数被提升为浮点数；第二组中，所有元素被统一成字符串。

可以用 `np.result_type()` 预测多个类型共同运算时的结果类型：

In [3]:
print(np.result_type(np.int32, np.float64))
print(np.result_type(np.int16, np.complex64))

float64
complex64


## 5. 使用 `astype()` 转换类型

`astype()` 会返回转换后的新数组，原数组不会被修改。

In [4]:
prices = np.array([12.8, 9.9, 20.1])
whole_numbers = prices.astype(np.int32)

print("原数组：", prices, prices.dtype)
print("转换后：", whole_numbers, whole_numbers.dtype)

原数组： [12.8  9.9 20.1] float64
转换后： [12  9 20] int32


> 浮点数转整数时会直接截掉小数部分，而不是四舍五入。需要四舍五入时，可以先使用 `np.round()`。

In [ ]:
values = np.array([1.2, 1.8, -1.8])

print("直接转整数：", values.astype(np.int32))
print("先四舍五入：", np.round(values).astype(np.int32))

转换前可以用 `np.can_cast()` 判断是否属于安全转换。所谓安全，是指目标类型能够保留源类型的所有可能取值。

In [5]:
print("int32 → int64：", np.can_cast(np.int32, np.int64, casting="safe"))
print("float64 → int32：", np.can_cast(np.float64, np.int32, casting="safe"))

int32 → int64： True
float64 → int32： False


## 6. 类型与内存占用

- `itemsize`：每个元素占用的字节数
- `nbytes`：数组全部元素占用的总字节数

通常有：`nbytes == size * itemsize`。

In [6]:
small = np.array([1, 2, 3, 4], dtype=np.int8)
large = np.array([1, 2, 3, 4], dtype=np.int64)

for name, array in [("int8 数组", small), ("int64 数组", large)]:
    print(name)
    print("  dtype   =", array.dtype)
    print("  itemsize=", array.itemsize, "字节")
    print("  nbytes  =", array.nbytes, "字节")

int8 数组
  dtype   = int8
  itemsize= 1 字节
  nbytes  = 4 字节
int64 数组
  dtype   = int64
  itemsize= 8 字节
  nbytes  = 32 字节


## 7. 常见陷阱：整数范围与溢出

整数类型的容量是有限的。例如，`int8` 只能保存 -128 到 127。超出范围的计算可能发生溢出，结果会绕回到范围内，而不是自动变成更大的整数类型。

In [7]:
for dtype in [np.int8, np.uint8, np.int16, np.int32]:
    info = np.iinfo(dtype)
    print(f"{dtype.__name__:6}：{info.min:12} 到 {info.max}")

int8  ：        -128 到 127
uint8 ：           0 到 255
int16 ：      -32768 到 32767
int32 ： -2147483648 到 2147483647


In [8]:
x = np.array([120], dtype=np.int8)

print("原值：", x)
print("120 + 10 的 int8 结果：", x + np.array([10], dtype=np.int8))
print("先转为 int16 再计算：", x.astype(np.int16) + 10)

原值： [120]
120 + 10 的 int8 结果： [-126]
先转为 int16 再计算： [130]


## 8. 常见陷阱：浮点数不是精确小数

二进制浮点数无法精确表示某些十进制小数。因此，不要用 `==` 直接判断浮点计算结果是否等于期望值；可以使用 `np.isclose()`。

In [ ]:
result = np.float64(0.1) + np.float64(0.2)

print("0.1 + 0.2 =", repr(result))
print("直接比较：", result == 0.3)
print("近似比较：", np.isclose(result, 0.3))

0.1 + 0.2 = np.float64(0.30000000000000004)
直接比较： False
近似比较： True


## 9. 字符串 dtype 中的长度

Unicode 字符串类型常显示为 `<U5` 等形式，其中 `U` 表示 Unicode，数字表示可容纳的字符数。固定长度可能导致后续赋值被截断。

In [10]:
labels = np.array(["cat", "dog"])
print("创建时：", labels, labels.dtype)

labels[0] = "elephant"
print("赋值后：", labels, labels.dtype)
print("elephant 被截断，是因为原数组只为每项预留了 3 个字符。")

创建时： ['cat' 'dog'] <U3
赋值后： ['ele' 'dog'] <U3
elephant 被截断，是因为原数组只为每项预留了 3 个字符。


## 10. 了解即可：结构化数据类型

菜鸟教程还展示了结构化 `dtype`。它允许一个元素包含多个有名字的字段，类似一行小型表格。真实数据分析中，通常也会使用 pandas 的 `DataFrame`；这里先了解 NumPy 的写法。

In [11]:
student_dtype = np.dtype([
    ("name", "U10"),
    ("age", np.int8),
    ("score", np.float32),
])

students = np.array([
    ("小明", 18, 92.5),
    ("小红", 19, 88.0),
], dtype=student_dtype)

print(students)
print("整个 dtype：", students.dtype)
print("姓名列：", students["name"])
print("成绩列：", students["score"])

[('小明', 18, 92.5) ('小红', 19, 88. )]
整个 dtype： [('name', '<U10'), ('age', 'i1'), ('score', '<f4')]
姓名列： ['小明' '小红']
成绩列： [92.5 88. ]


## 本课小结

1. `array.dtype` 查看数组的元素类型。
2. `np.array(..., dtype=...)` 在创建时指定类型。
3. `array.astype(...)` 创建转换类型后的新数组。
4. 普通数组会将所有元素统一成一种类型。
5. `itemsize` 和 `nbytes` 可以查看内存占用。
6. 选择较小类型能节省内存，但必须确认数值不会超出范围。
7. 浮点数比较推荐使用 `np.isclose()`。

## 本节练习

请先独立完成下面的代码。

要求：

1. 查看 `temperatures` 的原始类型。
2. 将它转换成 `float32`，保存为 `temperatures_f32`。
3. 输出转换后的 `dtype`、`itemsize` 和 `nbytes`。
4. 将摄氏温度转换为华氏温度，公式为：`华氏 = 摄氏 × 1.8 + 32`。
5. 用 `np.isclose()` 判断第一个华氏温度是否约等于 32。

In [23]:
temperatures = np.array([0, 18.5, 26.3, 37, -5.2])

# 请在这里完成练习


## 参考答案（完成练习后再看）

In [24]:
import numpy as np
print("原始 dtype:", temperatures.dtype)

temperatures_f32 = temperatures.astype(np.float32)
fahrenheit = temperatures_f32 * np.float32(1.8) + np.float32(32)

print("转换后 dtype:", temperatures_f32.dtype)
print("每个元素字节数：", temperatures_f32.itemsize)
print("数组总字节数：", temperatures_f32.nbytes)
print("华氏温度：", fahrenheit)
print("第一个值约等于 32:", np.isclose(fahrenheit[0], 32))

原始 dtype: float64
转换后 dtype: float32
每个元素字节数： 4
数组总字节数： 20
华氏温度： [32.   65.3  79.34 98.6  22.64]
第一个值约等于 32: True


---

下一课建议学习：[NumPy 数组属性](https://www.runoob.com/numpy/numpy-array-attributes.html)，重点掌握 `ndim`、`shape`、`size`、`itemsize` 和 `flags`。